# Fine-tune T5-small on ParaDetox
Train a model to rewrite toxic English comments into neutral language while preserving meaning.

**In Colab:** choose **Runtime → Change runtime type → GPU**, then run the cells in order. The first run downloads the model and dataset. GPU availability and training time vary.

Set `SMOKE_TEST = True` in the configuration cell for a quick end-to-end check before full training. A smoke-test model is only a debugging artifact, not a useful trained model.

Sources: [ParaDetox](https://huggingface.co/datasets/s-nlp/paradetox), [T5-small](https://huggingface.co/google-t5/t5-small), and [Hugging Face sequence-to-sequence training](https://huggingface.co/docs/transformers/tasks/translation).

## 1. Install dependencies
Use a fresh Colab runtime with Python 3.10 or newer. Colab provides PyTorch; install the remaining packages below. If Colab asks to restart the session, restart it and continue from the next cell. These versions use `eval_strategy` and `processing_class` in the Trainer API.

In [11]:
%pip install -q "transformers==4.57.3" "datasets==4.4.1" "accelerate==1.12.0" "sentencepiece==0.2.1"

## 2. Configure the run
These are the main settings to edit. The effective training batch size on one GPU is `8 × 2 = 16`. Full precision is used for both training and evaluation. All model parameters will be trained.

Smoke mode uses at most 128 training rows, 32 validation rows, 32 test rows, and 5 optimizer steps. Its output folders are separate from the full model.

In [12]:
import os
import random
import shutil
from pathlib import Path

import torch
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

MODEL_NAME = "google-t5/t5-small"
DATASET_NAME = "s-nlp/paradetox"
SOURCE_COLUMN = "en_toxic_comment"
TARGET_COLUMN = "en_neutral_comment"
PREFIX = "detoxify: "
SEED = 42
MAX_LENGTH = 128
EPOCHS = 5 #3
LEARNING_RATE = 3e-4
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
SMOKE_TEST = False
SMOKE_STEPS = 5

RUN_DIR = "./t5-small-paradetox-smoke-checkpoints" if SMOKE_TEST else "./epoch5-checkpoints"
EXPORT_DIR = "./t5-small-paradetox-smoke" if SMOKE_TEST else "./epoch5"
os.environ["WANDB_DISABLED"] = "true"
set_seed(SEED)
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Mode:", "smoke test" if SMOKE_TEST else "full training")
if not torch.cuda.is_available():
    print("For Colab training, switch to a GPU runtime. CPU training will be much slower.")

CUDA GPU: NVIDIA A100-SXM4-80GB
Mode: full training


## 3. Load and split the data
ParaDetox provides one `train` split. Each row pairs a toxic comment with a neutral rewrite. We create our own approximately 80%/10%/10% train/validation/test split.

Some toxic comments appear more than once with different rewrites. Split **unique input groups**, then assign their rows, to prevent leakage. Group keys ignore case and extra whitespace; the original text is preserved for training. The percentages apply to groups, so row counts can differ slightly. Validation selects the best checkpoint; test data is held back until training finishes.

In [13]:
raw = load_dataset(DATASET_NAME, split="train")
required_columns = {SOURCE_COLUMN, TARGET_COLUMN}
assert required_columns.issubset(raw.column_names), f"Missing columns: {required_columns - set(raw.column_names)}"

def valid_pair(row):
    return all(isinstance(row[column], str) and row[column].strip() for column in required_columns)

original_count = len(raw)
raw = raw.filter(valid_pair)
print(f"Loaded {original_count:,} rows; retained {len(raw):,} nonempty text pairs.")

def input_key(text):
    return " ".join(text.split()).casefold()

def grouped_split_indices(texts, seed=42):
    keys = [input_key(text) for text in texts]
    unique_keys = sorted(set(keys))
    if len(unique_keys) < 10:
        raise ValueError("Need at least 10 unique inputs for this split.")
    random.Random(seed).shuffle(unique_keys)
    train_end = int(0.8 * len(unique_keys))
    validation_end = int(0.9 * len(unique_keys))
    group_sets = {
        "train": set(unique_keys[:train_end]),
        "validation": set(unique_keys[train_end:validation_end]),
        "test": set(unique_keys[validation_end:]),
    }
    assert group_sets["train"].isdisjoint(group_sets["validation"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["validation"].isdisjoint(group_sets["test"])
    assignments = {key: name for name, group in group_sets.items() for key in group}
    indices = {name: [] for name in group_sets}
    for index, key in enumerate(keys):
        indices[assignments[key]].append(index)
    assert sum(map(len, indices.values())) == len(texts)
    assert all(indices.values())
    return indices

split_indices = grouped_split_indices(raw[SOURCE_COLUMN], SEED)
splits = DatasetDict({name: raw.select(indices) for name, indices in split_indices.items()})
if SMOKE_TEST:
    limits = {"train": 128, "validation": 32, "test": 32}
    splits = DatasetDict({
        name: data.shuffle(seed=SEED).select(range(min(len(data), limits[name])))
        for name, data in splits.items()
    })
print(splits)
print("Example input:", splits["train"][0][SOURCE_COLUMN])
print("Example target:", splits["train"][0][TARGET_COLUMN])

Loaded 19,744 rows; retained 19,744 nonempty text pairs.
DatasetDict({
    train: Dataset({
        features: ['en_toxic_comment', 'en_neutral_comment'],
        num_rows: 15787
    })
    validation: Dataset({
        features: ['en_toxic_comment', 'en_neutral_comment'],
        num_rows: 1985
    })
    test: Dataset({
        features: ['en_toxic_comment', 'en_neutral_comment'],
        num_rows: 1972
    })
})
Example input: he had steel balls too !
Example target: he was brave too!


## 4. Tokenize inputs and targets
T5 learns a text-to-text task: `detoxify: <toxic comment>` → `<neutral rewrite>`. Use the same prefix at inference time.

Tokenize inputs and targets separately and truncate each to 128 tokens. Do not pad the whole dataset here. The data collator pads each batch to its longest sequence and uses `-100` for target padding, which the loss ignores.

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
assert all(parameter.requires_grad for parameter in model.parameters())

def tokenize_batch(batch):
    inputs = [PREFIX + text for text in batch[SOURCE_COLUMN]]
    encoded = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    targets = tokenizer(text_target=batch[TARGET_COLUMN], max_length=MAX_LENGTH, truncation=True)
    encoded["labels"] = targets["input_ids"]
    return encoded

tokenized = splits.map(tokenize_batch, batched=True, remove_columns=raw.column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tokenized fields:", tokenized["train"].column_names)

Map:   0%|          | 0/1985 [00:00<?, ? examples/s]

Trainable parameters: 60,506,624
Tokenized fields: ['input_ids', 'attention_mask', 'labels']


## 5. Train and keep the best checkpoint
For a full run, evaluate and save after every epoch. The lowest validation loss determines the best checkpoint, which is loaded automatically at the end. Keep at most two checkpoints to limit disk usage.

In smoke mode, evaluate and save after the last of five steps instead. Checkpoints are separate from the final exported model. Re-running training starts a new run; use a new `RUN_DIR` if you want to preserve old checkpoints.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=RUN_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="steps" if SMOKE_TEST else "epoch",
    save_strategy="steps" if SMOKE_TEST else "epoch",
    eval_steps=SMOKE_STEPS if SMOKE_TEST else None,
    save_steps=SMOKE_STEPS if SMOKE_TEST else 500,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=1 if SMOKE_TEST else 50,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
)
train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss
1,1.021800,0.936315
2,0.898100,0.915603
3,0.853300,0.907596
4,0.752200,0.908328


## 6. Evaluate on held-out test data
Compute test loss only after checkpoint selection. This measures how well the model predicts reference rewrites; it does **not** independently measure toxicity reduction or meaning preservation. Review generated examples as well.

In [16]:
test_metrics = trainer.evaluate(eval_dataset=tokenized["test"], metric_key_prefix="test")
print(test_metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("test", test_metrics)
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

{'test_loss': 0.9065108895301819, 'test_runtime': 5.3435, 'test_samples_per_second': 369.045, 'test_steps_per_second': 46.224, 'epoch': 5.0}


## 7. Save and reload the model
Export the best model and its tokenizer. Reload from the saved directory to verify that inference works without the Trainer. The helper below handles one string at a time and returns an empty string for blank input. Longer inputs are truncated to the configured token limit.

In [17]:
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Reload on CPU to avoid holding a second model copy in GPU memory.
# For faster inference in a separate session, move this model to "cuda".
inference_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR)
inference_model = AutoModelForSeq2SeqLM.from_pretrained(EXPORT_DIR).to("cpu")
inference_model.eval()

@torch.inference_mode()
def detoxify(text):
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not text.strip():
        return ""
    inputs = inference_tokenizer(
        PREFIX + text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True
    ).to(inference_model.device)
    output_ids = inference_model.generate(
        **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
    )
    return inference_tokenizer.decode(output_ids[0], skip_special_tokens=True)

assert detoxify("   ") == ""
print("Saved and reloaded:", Path(EXPORT_DIR).resolve())

Saved and reloaded: /content/epoch5


## 8. Inspect rewrites and try your own text
Compare input, reference, and model output. Look for both a more neutral tone and preservation of the original meaning. References and model outputs can be imperfect. The first five test examples are a qualitative check, not a full quality evaluation.

In [18]:
for example in splits["test"].select(range(min(5, len(splits["test"])))):
    print("INPUT:    ", example[SOURCE_COLUMN])
    print("REFERENCE:", example[TARGET_COLUMN])
    print("MODEL:    ", detoxify(example[SOURCE_COLUMN]))
    print()

print("Your example:", detoxify("This is a stupid idea and you never listen."))

INPUT:     i want you all to myself , you sexy beast .
REFERENCE: i want you all to myself , honey .
MODEL:     i want you all to myself, you sexy beast.

INPUT:     if any of the cry babies who dont respect democracy come , it will be the stupid drain
REFERENCE: If any of the cry babies who dont respect democracy come , it will be the bad drain.
MODEL:     if any of the cry babies who dont respect democracy come, it will be the bad drain

INPUT:     time to pull this rubbish .
REFERENCE: Time to attend to this issue.
MODEL:     Time to pull this thing.

INPUT:     the election of van der bellen was a victory for stupidity and cultural suicide .
REFERENCE: the election of van der bellen was a victory for being unknowledgeable and cultural suicide .
MODEL:     the election of van der bellen was a victory for cultural suicide.

INPUT:     clinton is a loser , twice .
REFERENCE: Clinton lost twice.
MODEL:     clinton is a loser, twice.

Your example: This is a bad idea and you never liste

## 9. Download the model
Create a ZIP of the exported model and tokenizer, excluding training checkpoints. In Colab, the next cell starts a browser download. Download before disconnecting: files in the runtime are temporary.

To use the model later, unzip it, install the dependencies, and run the reload/helper code from section 7 (without the first two save lines). Set `EXPORT_DIR` to the extracted folder, `PREFIX = "detoxify: "`, and `MAX_LENGTH = 128`.

In [19]:
export_path = Path(EXPORT_DIR).resolve()
archive_path = shutil.make_archive(
    str(export_path), "zip", root_dir=export_path.parent, base_dir=export_path.name
)
print("Model archive:", archive_path)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: copy the ZIP from the path above.")
else:
    files.download(archive_path)

Model archive: /content/epoch5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/t5-small-paradetox",
    "zip",
    "/content",
    "t5-small-paradetox",
)
files.download("/content/t5-small-paradetox.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>